### Customers Data

**Load both Silver customer tables**

In [0]:
from pyspark.sql.functions import (
    col,
    concat,
    lit,
)

company_a_customers_df = spark.table(
    "workspace.silver.company_a_customers"
)

company_b_customers_df = spark.table(
    "workspace.silver.company_b_customers"
)

**Create canonical customer IDs**

**Company A**

In [0]:
company_a_customer_gold_df = (
    company_a_customers_df
    .select(
        col("customer_id").alias("source_customer_id"),
        col("customer_name"),
        col("email"),
        col("city"),
        col("state"),
        col("signup_date"),
    )
    .withColumn(
        "customer_id",
        concat(
            lit("A_"),
            col("source_customer_id")
        )
    )
    .withColumn(
        "source_system",
        lit("company_a")
    )
)

**Company B**

In [0]:
company_b_customer_gold_df = (
    company_b_customers_df
    .select(
        col("customer_id").alias("source_customer_id"),
        col("customer_name"),
        col("email"),
        col("city"),
        col("state"),
        col("signup_date"),
    )
    .withColumn(
        "customer_id",
        concat(
            lit("B_"),
            col("source_customer_id")
        )
    )
    .withColumn(
        "source_system",
        lit("company_b")
    )
)

In [0]:
dim_customer_df = (
    company_a_customer_gold_df
    .unionByName(
        company_b_customer_gold_df
    )
)

**Validate**

In [0]:
print(
    "Company A customers:",
    company_a_customer_gold_df.count()
)

print(
    "Company B customers:",
    company_b_customer_gold_df.count()
)

print(
    "Unified customers:",
    dim_customer_df.count()
)

In [0]:
(
    dim_customer_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
dim_customer_df.filter(
    col("customer_id").isNull()
).count()

**Write**

In [0]:
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_customer"
    )
)


### Products Data

**Load the unified product master**

In [0]:
unified_products_df = spark.table(
    "workspace.silver.unified_products"
)

In [0]:
print(
    "Unified product count:",
    unified_products_df.count()
)

unified_products_df.printSchema()

**build the Gold product dimension**

In [0]:
dim_product_df = (
    unified_products_df
    .select(
        col("canonical_product_id")
            .alias("product_id"),

        col("product_name"),
        col("category"),
        col("unit_price")
            .alias("catalog_unit_price"),

        col("source_system")
    )
)

In [0]:
print(
    "Gold products:",
    dim_product_df.count()
)

In [0]:
(
    dim_product_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
dim_product_df.filter(
    col("product_id").isNull()
).count()

**Write**

In [0]:
(
    dim_product_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_product"
    )
)

**fact_sales**

**Load company A orders and orders items silvers**

In [0]:
company_a_orders_df = spark.table(
    "workspace.silver.company_a_orders"
)

company_a_order_items_df = spark.table(
    "workspace.silver.company_a_order_items"
)

In [0]:
company_a_fact_df = (
    company_a_order_items_df.alias("oi")
    .join(
        company_a_orders_df.alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .select(
        col("oi.order_item_id"),
        col("oi.order_id"),

        concat(
            lit("A_"),
            col("o.customer_id")
        ).alias("customer_id"),

        col("oi.product_id").alias("product_id"),

        col("o.order_date"),
        col("o.order_status"),
        col("o.payment_method"),

        col("oi.quantity"),
        col("oi.unit_price"),
        col("oi.discount_pct"),

        lit("company_a").alias("source_system")
    )
)

**Load company B orders and orders items silvers**

In [0]:
company_b_orders_df = spark.table(
    "workspace.silver.company_b_orders"
)

company_b_order_items_df = spark.table(
    "workspace.silver.company_b_order_items_canonical"
)

In [0]:
company_b_fact_df = (
    company_b_order_items_df.alias("oi")
    .join(
        company_b_orders_df.alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .select(
        col("oi.order_item_id"),
        col("oi.order_id"),

        concat(
            lit("B_"),
            col("o.customer_id")
        ).alias("customer_id"),

        col("oi.product_id").alias("product_id"),

        col("o.order_date"),
        col("o.order_status"),
        col("o.payment_method"),

        col("oi.quantity"),
        col("oi.unit_price"),
        col("oi.discount_pct"),

        lit("company_b").alias("source_system")
    )
)

**Combine**

In [0]:
fact_sales_base_df = (
    company_a_fact_df
    .unionByName(
        company_b_fact_df
    )
)

In [0]:
print(
    "Company A fact rows:",
    company_a_fact_df.count()
)

print(
    "Company B fact rows:",
    company_b_fact_df.count()
)

print(
    "Combined fact rows:",
    fact_sales_base_df.count()
)

**Calculate revenue measures**

In [0]:
fact_sales_df = (
    fact_sales_base_df
    .withColumn(
        "gross_amount",
        col("quantity") * col("unit_price")
    )
    .withColumn(
        "discount_amount",
        (
            col("quantity")
            * col("unit_price")
            * col("discount_pct")
        ) / lit(100)
    )
    .withColumn(
        "net_amount",
        col("gross_amount")
        - col("discount_amount")
    )
)

**Revenue status rule**

In [0]:
from pyspark.sql.functions import when

fact_sales_df = (
    fact_sales_df
    .withColumn(
        "recognized_revenue",
        when(
            col("order_status") == "Completed",
            col("net_amount")
        ).otherwise(lit(0))
    )
)

**Validate foreign keys**

In [0]:
dim_customer_df = spark.table(
    "workspace.gold.dim_customer"
)

invalid_customer_refs_df = (
    fact_sales_df.alias("f")
    .join(
        dim_customer_df
        .select("customer_id")
        .alias("c"),
        col("f.customer_id")
        == col("c.customer_id"),
        "left_anti"
    )
)

print(
    "Invalid customer references:",
    invalid_customer_refs_df.count()
)

In [0]:
dim_product_df = spark.table(
    "workspace.gold.dim_product"
)

invalid_product_refs_df = (
    fact_sales_df.alias("f")
    .join(
        dim_product_df
        .select("product_id")
        .alias("p"),
        col("f.product_id")
        == col("p.product_id"),
        "left_anti"
    )
)

print(
    "Invalid product references:",
    invalid_product_refs_df.count()
)

**Write**

In [0]:
(
    fact_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.fact_sales"
    )
)

In [0]:
spark.sql("""
SELECT
    source_system,
    COUNT(*) AS rows,
    ROUND(SUM(recognized_revenue), 2) AS revenue
FROM workspace.gold.fact_sales
GROUP BY source_system
ORDER BY source_system
""").show()

**KPI / analytics layer**

In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    countDistinct,
    year,
    month,
    when,
    lit,
)

fact_sales_df = spark.table(
    "workspace.gold.fact_sales"
)

order_summary_df = (
    fact_sales_df
    .groupBy(
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
        "payment_method",
        "source_system",
    )
    .agg(
        sum("gross_amount").alias("gross_amount"),
        sum("discount_amount").alias("discount_amount"),
        sum("net_amount").alias("net_amount"),
        sum("recognized_revenue").alias("recognized_revenue"),
    )
)

In [0]:
print(
    "Fact rows:",
    fact_sales_df.count()
)

print(
    "Unique orders:",
    order_summary_df.count()
)

In [0]:
order_summary_df = (
    order_summary_df
    .withColumn(
        "year",
        year(col("order_date"))
    )
    .withColumn(
        "month",
        month(col("order_date"))
    )
)

In [0]:
monthly_kpi_df = (
    order_summary_df
    .groupBy(
        "year",
        "month",
        "source_system",
    )
    .agg(
        countDistinct("order_id")
            .alias("total_orders"),

        countDistinct(
            when(
                col("order_status") == "Completed",
                col("order_id")
            )
        ).alias("completed_orders"),

        countDistinct(
            when(
                col("order_status") == "Cancelled",
                col("order_id")
            )
        ).alias("cancelled_orders"),

        countDistinct(
            when(
                col("order_status") == "Returned",
                col("order_id")
            )
        ).alias("returned_orders"),

        sum("recognized_revenue")
            .alias("recognized_revenue"),
    )
)

In [0]:
monthly_kpi_df = (
    monthly_kpi_df
    .withColumn(
        "average_order_value",
        when(
            col("completed_orders") > 0,
            col("recognized_revenue")
            / col("completed_orders")
        ).otherwise(lit(0))
    )
)

In [0]:
monthly_kpi_df.orderBy(
    "year",
    "month",
    "source_system"
).show(
    100,
    truncate=False
)

**Write the KPI**

In [0]:
(
    monthly_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.kpi_monthly_sales"
    )
)

In [0]:
spark.sql("""
SELECT *
FROM workspace.gold.kpi_monthly_sales
ORDER BY year, month, source_system
""").show(100, truncate=False)

**category performance KPI table**

In [0]:
fact_sales_df = spark.table(
    "workspace.gold.fact_sales"
)

dim_product_df = spark.table(
    "workspace.gold.dim_product"
)

sales_with_product_df = (
    fact_sales_df.alias("f")
    .join(
        dim_product_df.alias("p"),
        col("f.product_id") == col("p.product_id"),
        "inner"
    )
)

In [0]:
from pyspark.sql.functions import (
    sum,
    countDistinct,
    avg,
)

category_kpi_df = (
    sales_with_product_df
    .groupBy(
        col("p.category").alias("category"),
        col("f.source_system").alias("source_system"),
    )
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        sum("f.quantity").alias("units_sold"),
        sum("f.gross_amount").alias("gross_sales"),
        sum("f.discount_amount").alias("discount_amount"),
        sum("f.recognized_revenue").alias("recognized_revenue"),
        avg("f.discount_pct").alias("avg_discount_pct"),
    )
)

In [0]:
category_kpi_df.orderBy(
    col("recognized_revenue").desc()
).show(
    100,
    truncate=False
)

**write**

In [0]:
(
    category_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.kpi_category_performance"
    )
)

In [0]:
product_kpi_df = (
    sales_with_product_df
    .groupBy(
        col("f.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("f.source_system"),
    )
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        sum("f.quantity").alias("units_sold"),
        sum("f.recognized_revenue").alias("recognized_revenue"),
    )
)

In [0]:
(
    product_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.kpi_product_performance"
    )
)

**Company A vs Company B merger comparison**

In [0]:
fact_sales_df = spark.table(
    "workspace.gold.fact_sales"
)

order_summary_df = (
    fact_sales_df
    .groupBy(
        "order_id",
        "source_system",
        "order_status",
    )
    .agg(
        sum("recognized_revenue")
            .alias("recognized_revenue")
    )
)

In [0]:
from pyspark.sql.functions import (
    countDistinct,
    sum,
    when,
    col,
    lit,
)

company_comparison_df = (
    order_summary_df
    .groupBy("source_system")
    .agg(
        countDistinct("order_id")
            .alias("total_orders"),

        countDistinct(
            when(
                col("order_status") == "Completed",
                col("order_id")
            )
        ).alias("completed_orders"),

        countDistinct(
            when(
                col("order_status") == "Cancelled",
                col("order_id")
            )
        ).alias("cancelled_orders"),

        countDistinct(
            when(
                col("order_status") == "Returned",
                col("order_id")
            )
        ).alias("returned_orders"),

        sum("recognized_revenue")
            .alias("recognized_revenue"),
    )
)

In [0]:
company_comparison_df = (
    company_comparison_df
    .withColumn(
        "average_order_value",
        when(
            col("completed_orders") > 0,
            col("recognized_revenue")
            / col("completed_orders")
        ).otherwise(lit(0))
    )
)

In [0]:
company_comparison_df = (
    company_comparison_df
    .withColumn(
        "cancellation_rate_pct",
        (
            col("cancelled_orders")
            / col("total_orders")
        ) * 100
    )
    .withColumn(
        "return_rate_pct",
        (
            col("returned_orders")
            / col("total_orders")
        ) * 100
    )
)

In [0]:
company_comparison_df.show(
    truncate=False
)

**write**

In [0]:
(
    company_comparison_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.kpi_company_comparison"
    )
)

**customer performance**

In [0]:
fact_sales_df = spark.table(
    "workspace.gold.fact_sales"
)

dim_customer_df = spark.table(
    "workspace.gold.dim_customer"
)

In [0]:
sales_with_customer_df = (
    fact_sales_df.alias("f")
    .join(
        dim_customer_df.alias("c"),
        col("f.customer_id") == col("c.customer_id"),
        "inner"
    )
)

In [0]:
from pyspark.sql.functions import (
    sum,
    countDistinct,
    max,
    min,
)

customer_kpi_df = (
    sales_with_customer_df
    .groupBy(
        col("f.customer_id"),
        col("c.customer_name"),
        col("c.city"),
        col("c.state"),
        col("f.source_system"),
    )
    .agg(
        countDistinct("f.order_id")
            .alias("total_orders"),

        sum("f.quantity")
            .alias("total_units"),

        sum("f.recognized_revenue")
            .alias("lifetime_revenue"),

        min("f.order_date")
            .alias("first_order_date"),

        max("f.order_date")
            .alias("last_order_date"),
    )
)

In [0]:
from pyspark.sql.functions import when, lit

customer_kpi_df = (
    customer_kpi_df
    .withColumn(
        "average_order_value",
        when(
            col("total_orders") > 0,
            col("lifetime_revenue")
            / col("total_orders")
        ).otherwise(lit(0))
    )
)

In [0]:
customer_kpi_df.orderBy(
    col("lifetime_revenue").desc()
).show(
    50,
    truncate=False
)

**write**

In [0]:
(
    customer_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.kpi_customer_performance"
    )
)

**final Gold validation pass**

In [0]:
# 1. Dimension counts
print(
    "dim_customer:",
    spark.table("workspace.gold.dim_customer").count()
)

print(
    "dim_product:",
    spark.table("workspace.gold.dim_product").count()
)

print(
    "fact_sales:",
    spark.table("workspace.gold.fact_sales").count()
)

In [0]:
from pyspark.sql.functions import col

duplicate_customers = (
    spark.table("workspace.gold.dim_customer")
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

duplicate_products = (
    spark.table("workspace.gold.dim_product")
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate customers:", duplicate_customers)
print("Duplicate products:", duplicate_products)

In [0]:
fact_df = spark.table(
    "workspace.gold.fact_sales"
)

customer_df = spark.table(
    "workspace.gold.dim_customer"
)

product_df = spark.table(
    "workspace.gold.dim_product"
)

In [0]:
invalid_product_fk = (
    fact_df.alias("f")
    .join(
        product_df.select("product_id").alias("p"),
        col("f.product_id") == col("p.product_id"),
        "left_anti"
    )
    .count()
)

print(
    "Invalid fact product references:",
    invalid_product_fk
)

In [0]:
fact_df.filter(
    (col("gross_amount") < 0)
    | (col("discount_amount") < 0)
    | (col("net_amount") < 0)
    | (col("recognized_revenue") < 0)
).count()

In [0]:
spark.sql("""
SHOW TABLES IN workspace.gold
""").show(truncate=False)